### nanoGPT

### Embeddings
- need to encode tokens (chars/words/etc.) into numerical values
- tradeoff of vocab size and sequence length
- vocab size = # of unique tokens. 
- More unique tokens => longer token size = smaller sequence lengths 
- 24 chars in alphabet. 1 chars = 24 unique tokens. 2 chars = 24 * 24 unique tokens

### Training sequences
- goal: predict next token
- target = shift input by 1

ex. 
input: the dog runs to the fence  

output: dog runs to the fence ____  

- when input is tensor([18]) the target: 47
- when input is tensor([18, 47]) the target: 56
- when input is tensor([18, 47, 56]) the target: 57
- when input is tensor([18, 47, 56, 57]) the target: 58
- when input is tensor([18, 47, 56, 57, 58]) the target: 1
- when input is tensor([18, 47, 56, 57, 58,  1]) the target: 15
- when input is tensor([18, 47, 56, 57, 58,  1, 15]) the target: 47
- when input is tensor([18, 47, 56, 57, 58,  1, 15, 47]) the target: 58


QKV tokens
- Q = what i'm looking for
- K = what I contain
- V = what I'm communicating, if I am important to you (high Q @ K)

- Q @ K - how much my query for my token "attends" to the "keys/contents" of other tokens

In [1]:
import torch
import torch.nn as nn
from torch.nn import functional as F

In [ ]:
# self attention

# version 4: self-attention!
torch.manual_seed(1337)
B,T,C = 4,8,32 # batch, time/sequence, channels
x = torch.randn(B,T,C)

# let's see a single Head perform self-attention
head_size = 16 # embedding dimension

key = nn.Linear(C, head_size, bias=False)
query = nn.Linear(C, head_size, bias=False)
value = nn.Linear(C, head_size, bias=False)

k = key(x)   # (B, T, 16)
q = query(x) # (B, T, 16)
wei =  q @ k.transpose(-2, -1) # (B, T, 16) @ (B, 16, T) ---> (B, T, T) // table of self attention weights

tril = torch.tril(torch.ones(T, T))
#wei = torch.zeros((T,T))
wei = wei.masked_fill(tril == 0, float('-inf')) # masked self-attention w/ -inf for softmax
wei = F.softmax(wei, dim=-1) # distribution on unmasked portions (row-wise across the sequence)

v = value(x)
out = wei @ v # dot product / upweight by value matrix
# wei = (B,T,T)
# v = (B, T, Embedding size)

#out = wei @ x

out.shape # (B, T, embedding_dim) -> same as KQV's size!

torch.Size([4, 8, 16])

In [ ]:
wei[0] # one batch (B, T, T) - table of self-attention weights

# rows are sequences

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1574, 0.8426, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2088, 0.1646, 0.6266, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5792, 0.1187, 0.1889, 0.1131, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0294, 0.1052, 0.0469, 0.0276, 0.7909, 0.0000, 0.0000, 0.0000],
        [0.0176, 0.2689, 0.0215, 0.0089, 0.6812, 0.0019, 0.0000, 0.0000],
        [0.1691, 0.4066, 0.0438, 0.0416, 0.1048, 0.2012, 0.0329, 0.0000],
        [0.0210, 0.0843, 0.0555, 0.2297, 0.0573, 0.0709, 0.2423, 0.2391]],
       grad_fn=<SelectBackward0>)

Notes:
- Attention is a **communication mechanism**. Can be seen as nodes in a directed graph looking at each other and aggregating information with a weighted sum from all nodes that point to them, with data-dependent weights.
- There is no notion of space. Attention simply acts over a set of vectors. This is why we need to positionally encode tokens.
- Each example across batch dimension is of course processed completely independently and never "talk" to each other
- In an "encoder" attention block just delete the single line that does masking with `tril`, allowing all tokens to communicate. This block here is called a "decoder" attention block because it has triangular masking, and is usually used in autoregressive settings, like language modeling.
- "self-attention" just means that the keys and values are produced from the same source as queries. In "cross-attention", the queries still get produced from x, but the keys and values come from some other, external source (e.g. an encoder module)
- "Scaled" attention additional divides `wei` by 1/sqrt(head_size). This makes it so when input Q,K are unit variance, wei will be unit variance too and Softmax will stay diffuse and not saturate too much. Illustration below

Summary
1. Attention is a a graph where we aggregate all nodes pointing at a single node as a linear combo. The other nodes "attend" to our node of focus which gives different levels of attention to the other nodes.
2. No notion of space or order. it needs to be positionally encoded.
3. Attention is applied across the sequence dim. for each unit in batch. Each unit is independent of one another.
4. Masked attention is optional. Masked = Decoder for auto-regression. w/o Masked = encoder.
5. "self-attention" = same source for Keys/Values and Queries. "cross-attention" - keys/values from x and queries are from another source.
6. 

In [ ]:
k = torch.randn(B,T,head_size) # unit gaussian
q = torch.randn(B,T,head_size) # unit gaussian 
wei = q @ k.transpose(-2, -1) * head_size**-0.5 # scale it back down so softmax diffuses and doesn't become peaky

# wei -> softmax

k.var(), q.var(), wei.var()

(tensor(1.0449), tensor(1.0700), tensor(1.0918))

In [ ]:
torch.softmax(torch.tensor([0.1, -0.2, 0.3, -0.2, 0.5]), dim=-1) # softmax with spread out values

tensor([0.1925, 0.1426, 0.2351, 0.1426, 0.2872])

In [6]:
torch.softmax(torch.tensor([0.1, -0.2, 0.3, -0.2, 0.5])*8, dim=-1) # gets too peaky, converges to one-hot

tensor([0.0326, 0.0030, 0.1615, 0.0030, 0.8000])

### Multiheaded attention
- channel size = head_size/# of heads
- so you kind of group by heads and then concat them all together at the end to get the full channel size

### Layernorm
- applied before ffwd not after

In [8]:
class LayerNorm1d: # (used to be BatchNorm1d)

  def __init__(self, dim, eps=1e-5, momentum=0.1):
    self.eps = eps
    self.gamma = torch.ones(dim)
    self.beta = torch.zeros(dim)

  def __call__(self, x):
    # calculate the forward pass
    xmean = x.mean(1, keepdim=True) # batch mean
    xvar = x.var(1, keepdim=True) # batch variance
    xhat = (x - xmean) / torch.sqrt(xvar + self.eps) # normalize to unit variance
    self.out = self.gamma * xhat + self.beta
    return self.out

  def parameters(self):
    return [self.gamma, self.beta]

torch.manual_seed(1337)
module = LayerNorm1d(100)
x = torch.randn(32, 100) # batch size 32 of 100-dimensional vectors
x = module(x)
x.shape

torch.Size([32, 100])

In [ ]:
# hyperparameters
batch_size = 16 # how many independent sequences will we process in parallel?
block_size = 32 # what is the maximum context length for predictions?
max_iters = 1 # 5000
eval_interval = 100
learning_rate = 1e-3
device = 'cuda' if torch.cuda.is_available() else 'cpu'
eval_iters = 200
n_embd = 64
n_head = 4
n_layer = 4
dropout = 0.0

# ---

# helper functions 

torch.manual_seed(1337)

# wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

# positional embedding
# here are all the unique characters that occur in this text
chars = sorted(list(set(text)))
vocab_size = len(chars)
# create a mapping from characters to integers
stoi = { ch:i for i,ch in enumerate(chars) } # position is just order in alphabet
itos = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [stoi[c] for c in s] # encoder: take a string, output a list of integers
decode = lambda l: ''.join([itos[i] for i in l]) # decoder: take a list of integers, output a string

# Train and test splits
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9*len(data)) # first 90% will be train, rest val
train_data = data[:n]
val_data = data[n:]


# data loading
def get_batch(split):
    # generate a small batch of data of inputs x and targets y
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y


@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

# Model

class Head(nn.Module):
    """ one head of self-attention """

    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False) # head size = embedding dimension
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size))) # masked attention (decoder)

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B,T,C = x.shape
        k = self.key(x)   # (B,T,C)
        q = self.query(x) # (B,T,C)
        # compute attention scores ("affinities")
        wei = q @ k.transpose(-2,-1) * C**-0.5 # (B, T, C) @ (B, C, T) -> (B, T, T) // scale logits for unit variance before softmax
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf')) # (B, T, T)
        wei = F.softmax(wei, dim=-1) # (B, T, T)
        wei = self.dropout(wei)
        # perform the weighted aggregation of the values
        v = self.value(x) # (B,T,C)
        out = wei @ v # (B, T, T) @ (B, T, C) -> (B, T, C)
        return out


class MultiHeadAttention(nn.Module):
    """ multiple heads of self-attention in parallel """

    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        # get output of each head and concat on channel/embedding dimension (= full embedding dim from split across heads originally)
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.dropout(self.proj(out))
        return out

class FeedFoward(nn.Module):
    """ a simple linear layer followed by a non-linearity """

    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd), # middle hidden dim. scaled up according to original paper
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout), # to prevent overfitting
        )

    def forward(self, x):
        return self.net(x)


class Block(nn.Module):
    """ Transformer block: communication followed by computation """

    def __init__(self, n_embd, n_head):
        # n_embd: embedding dimension, n_head: the number of heads we'd like
        super().__init__()
        head_size = n_embd // n_head # get head size for each head in multi-head self attention. Full # of embedding dims split across heads
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedFoward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x)) # layer norm applied before layers (differs from original paper)
        x = x + self.ffwd(self.ln2(x))
        return x


# super simple bigram model
class BigramLanguageModel(nn.Module):

    def __init__(self):
        super().__init__()
        # each token directly reads off the logits for the next token from a lookup table
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd) # tokenize
        self.blocks = nn.Sequential(*[Block(n_embd, n_head=n_head) for _ in range(n_layer)]) # ! uses MSA blocks
        self.ln_f = nn.LayerNorm(n_embd) # final layer norm
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape

        # idx and targets are both (B,T) tensor of integers
        tok_emb = self.token_embedding_table(idx) # (B,T,C)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device)) # (T,C) // broadcast position embeddings
        x = tok_emb + pos_emb # (B,T,C)
        x = self.blocks(x) # (B,T,C)
        x = self.ln_f(x) # (B,T,C)
        logits = self.lm_head(x) # (B,T,vocab_size)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets) # cross entropy on predicted token logits and target token (in vocab table)

            # so logits = probs on the vocab table? 

        return logits, loss

    def generate(self, idx, max_new_tokens):
        # idx is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            # crop idx to the last block_size tokens
            idx_cond = idx[:, -block_size:]
            # get the predictions
            logits, loss = self(idx_cond)
            # focus only on the last time step
            logits = logits[:, -1, :] # becomes (B, C)
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)
            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx

model = BigramLanguageModel()
m = model.to(device)
# print the number of parameters in the model
print(sum(p.numel() for p in m.parameters())/1e6, 'M parameters')


0.209729 M parameters


In [10]:

# create a PyTorch optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

for iter in range(max_iters):

    # every once in a while evaluate the loss on train and val sets
    if iter % eval_interval == 0 or iter == max_iters - 1:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    # sample a batch of data
    xb, yb = get_batch('train')

    # evaluate the loss
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

# generate from the model
context = torch.zeros((1, 1), dtype=torch.long, device=device)
print(decode(m.generate(context, max_new_tokens=2000)[0].tolist()))

step 0: train loss 4.4112, val loss 4.4015

I$jBnWodf;dc$dnVQMm
$;nc.BcJCNFrHlA'HI;KlAcr$mw,Hd w-f$3.JO:NRcFAJe.uoJEdZ.HgQXJiNUPtc-vfp?s CPlr3?gbzwNcUUfocHpZzgLc!Lz,:WfBHRZ.Vbm-MO
V.zE!F.?K?y3QigZO:3?hPm.Nv-HuwalTT$?k,pMv Ue'h&!begza-rop&cPdhRO&&Gf;sOARE;FK,NpxegcDsKR;s$DUQhRcJe!C'&fLo3Vs.$&pJ!hXyVdudJ:3$!Dke&.-!azNGbqcVVdlf;C&-OjjUPLKsDFk,
VFRL.$cIjwcXCdYbIvCmDi$aTXgr?YGzIXugphbzpVFDw:Yo&DxginoqFNriy3&irhNbANhbfU3.w&:zn?l &?RMeYTMtUI'p O&zNJXoaryQiMXo,mMTWKAJbmu3?pLrbS enL&J!joujaBeDXryt.WCg
vFlS?OmDh3dzqJiEypYGe?f-zNzRQVm3SDktEjXbtghqetoa;'cfRxhZwXcdMNnVK'pXyLXzlAeNxif$p!z'crpiKy CPXvLiNSRgXTgFL.ascX.q,s?QbitR'fHpbOjgQuRnc,p gQVpUtwUrrlcENeyXengiH&lhqKOm$bc,RbeYMikJTXJCEW&$b,VNoDngU,PNcp!D;KGbSVw-wb.Heyc;RZ!SA
c33,a QJC:MuXcfh,GiHXQ:cp:;etftZOk3fjVDpK?whboE;Ooio;ehYfB
-R,bDDosk,lhXpdDCpqUxVrio&Z-OmRWfyU'FFoob

rTUmHsl'ongqUp&zauStApJQ'AR&f3Ttvsos:whcAtcWtL.mohs!Y&?:fuF,jzRO
VjeiKCWZnE$syUpp3FU-'?pimhiJQJjVroAeHDowHtghDf
UQ?zZWLd$Kn!C$  HbK fq;e,YTOaLEXJDzh?EpLX?Hh$dDXX3DQuwz-QcPf.ihN

Karpathy working implementation ( 15 mins on A100 that I don't have :( ) - https://colab.research.google.com/drive/1JMLa53HDuA-i7ZBmqV7ZnA3c_fvtXnx-?usp=sharing#scrollTo=hoelkOrFY8bN